Intro:
- Structured streaming was introduced in recent version of API
- Replaces old syntax for streaming
- Structured streaming uses Dataframes

Examples:
- sum of numbers received from netcat
- sum of numbers received from netcat windowed into groups of 10 seconds
- streaming data from Twitter

Spark Dataframes can receive input from a stream of data using a feature called structured streaming. Structured streaming makes it relatively straightforward to use a stream for a Dataframes source, only requiring a few lines of code. Once the stream is setup, the Dataframe has all of the same features as any other Dataframe, meaning you can apply the same transformations to the streamed data.



# Streaming from the Netcat

In this example I'll use Netcat to send sales for different items. My PySpark program will read these messages as a stream and create a sum of the numbers.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, sum

spark = SparkSession \
    .builder \
    .appName('Streaming Example') \
    .getOrCreate()
    
lines = spark \
    .readStream \
    .format('socket') \
    .option('host', 'localhost') \
    .option('port, 9999) \
    .load()

sales = lines \
    .select(
       split(lines.value, ' ').alias('sale')
    )
    
sales = sales \
    .withColumn('item', sales.sale.getItem(0)) \
    .withColumn('cost', sales.sale.getItem(1))

item_sales = sales \
    .groupBy("item") \
    .agg(sum('cost'))

query = item_sales \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()

The sales information sent through netcat will look like this:

``` command-line
jumper 10
paper 5
paper 1
socks 3
jumper 5
```

The item name and it's price is separated by a space.

Before I run the Python code I start up netcat with this command:

``` command-line
nc -lk 9999 
```

Now I start the Python program and enter the sales data one at a time into netcat.

This is the output for the Python program:
``` command-line
18/02/23 16:32:53 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.
-------------------------------------------
Batch: 0
-------------------------------------------
+------+---------+                                                              
|  item|sum(cost)|
+------+---------+
|jumper|     10.0|
+------+---------+

-------------------------------------------
Batch: 1
-------------------------------------------
+------+---------+                                                              
|  item|sum(cost)|
+------+---------+
|jumper|     10.0|
| paper|      5.0|
+------+---------+

-------------------------------------------
Batch: 2
-------------------------------------------
+------+---------+                                                              
|  item|sum(cost)|
+------+---------+
|jumper|     10.0|
| paper|      6.0|
+------+---------+

-------------------------------------------
Batch: 3
-------------------------------------------
+------+---------+                                                              
|  item|sum(cost)|
+------+---------+
|jumper|     10.0|
| paper|      6.0|
| socks|      3.0|
+------+---------+

-------------------------------------------
Batch: 4
-------------------------------------------
+------+---------+                                                              
|  item|sum(cost)|
+------+---------+
|jumper|     15.0|
| paper|      6.0|
| socks|      3.0|
+------+---------+


```

There are several modes for output. In the above example I'm using `'complete'` output mode to show all of the data every time a new item is added. There is also an `'update'` output mode that will only show the records that are updated and **`'append'` output mode that does something.**

Spark will terminate the job when netcat is closed.

# Streaming with Windowed Output



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, sum, window, unix_timestamp, current_timestamp

spark = SparkSession \
    .builder \
    .appName('Windowed Streaming Example') \
    .getOrCreate()
    
lines = spark \
    .readStream \
    .format('socket') \
    .option('host', 'localhost') \
    .option('port', 9999) \
    .load()

sales = lines \
    .select(
       split(lines.value, ' ').alias('sale')
    )
    
sales = sales \
    .withColumn('item', sales.sale.getItem(0)) \
    .withColumn('cost', sales.sale.getItem(1)) \
    .withColumn('timestamp', current_timestamp())

item_sales = sales \
    .groupBy(
        window(sales.timestamp, "30 seconds", "30 seconds"),
        sales.item
    ) \
    .agg(sum('cost'))

query = item_sales \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()